# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL, making its metadata and structure both FAIR and machine navigable.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Review the available record sets and their field/column `@id`s. We will identify the record sets and their field structure using the Croissant metadata.

In [ ]:
# List available record sets and their fields (by @id)
record_sets = list(dataset.metadata.record_sets)
print("Available record sets [@id]:\n--------------------------")
for rs in record_sets:
    print(f"@id: {rs.id}; name: {rs.name}; description: {getattr(rs, 'description', '')}")
    print("  Fields/columns (@id):")
    for f in getattr(rs, 'fields', []):
        print(f"    - {f.id}; name: {f.name}; dataType: {f.data_type}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Collect data from all or selected record sets
dataframes = {}
selected_record_sets = [rs.id for rs in record_sets]  # Extract all, or manually pick '@id's

for rs_id in selected_record_sets:
    # Each record set yields a stream of records (as dicts)
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df

# For demonstration, show the columns of the first record set
example_rs_id = selected_record_sets[0]
print(f"Columns in record set {example_rs_id}:")
print(dataframes[example_rs_id].columns.tolist())
dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's process a numeric field for demonstration. We'll filter the main protein table for high-abundance proteins, normalize a numeric field, and group by a category if present.

In [ ]:
# Pick a relevant record set: first one found
rs_id = selected_record_sets[0]
df = dataframes[rs_id]

# Show available field/column IDs
print(f"Available columns in {rs_id}:")
print(df.columns.tolist())

# Try to identify numeric fields (float/int)
numeric_candidates = df.select_dtypes(include=["number"]).columns.tolist()
if not numeric_candidates:
    # If all fields imported as string, try convert a column
    print("Warning: No numeric columns auto-detected. Trying to cast columns to find numeric fields.")
    for col in df.columns:
        try:
            df[col + "_cast"] = pd.to_numeric(df[col], errors="coerce")
            if df[col + "_cast"].notna().sum() > 0:
                numeric_candidates.append(col + "_cast")
                print(f"{col} -> {col}_cast might be numeric.")
        except Exception:
            pass

# Select a numeric field based on inspection, e.g., 'Coverage (%)' or similar (change below as needed)
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = None
    print("No numeric field found.")

# Threshold filtering example
if numeric_field is not None:
    threshold = 10
    filtered_df = df[pd.to_numeric(df[numeric_field], errors="coerce") > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field], errors="coerce") - pd.to_numeric(filtered_df[numeric_field], errors="coerce").mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors="coerce").std()
    print(f"Normalized {numeric_field} example:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Pick a group-by field (if available, e.g. 'Description' or 'Sample label'), or skip
    group_candidates = [c for c in df.columns if c != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        if group_field in filtered_df.columns:
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
            )
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable group-by field found.")
else:
    print("No numeric field available for EDA.")

## 5. Visualization

Visualize distributions or relationships for the extracted numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show a histogram and boxplot for the normalized numeric field (if analysis successful)
if numeric_field is not None and f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30)
    plt.title(f"Distribution of normalized {numeric_field}")
    plt.xlabel(f"{numeric_field} [normalized]")
    
    plt.subplot(1,2,2)
    sns.boxplot(y=filtered_df[f"{numeric_field}_normalized"].dropna())
    plt.title(f"Boxplot: normalized {numeric_field}")
    plt.show()
else:
    print("Numeric field was not detected or extracted for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset described by a Croissant schema directly from its source, explored its structural metadata, reviewed the record sets, and extracted and analyzed tabular protein data. With a focus on reproducible and transparent workflows, we applied standard data processing (filtering, normalization, grouping) and explored numeric field distributions. See dataset documentation and the Croissant schema for further metadata-driven analysis using field IDs, and adapt code above for deeper domain-specific investigations.